## preprocessing pour normaliser

In [ ]:
import json
from pathlib import Path

from preprocessing_stylo import preprocess_corpus

In [ ]:
def load_book_json(path: Path) -> list[str]:
    """
    Charge un livre JSON (chapters / paragraphs)
    et retourne une liste de paragraphes.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    paragraphs = []
    for chapter in data.get("chapters", []):
        paragraphs.extend(chapter.get("paragraphs", []))

    return paragraphs

def load_books_from_dir(dir_path: str) -> dict:
    """
    Charge tous les livres JSON d'un dossier.
    Retourne un dict {nom_fichier: [paragraphes]}
    """
    dir_path = Path(dir_path)
    books = {}

    for json_file in dir_path.glob("*.json"):
        books[json_file.stem] = load_book_json(json_file)

    return books

In [ ]:
books_sarkozy = load_books_from_dir("/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/livres_sarkozy")
books_bardella = load_books_from_dir("/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/livres_bardella")
with open("/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/whisper_transcriptions_merged.json", encoding="utf-8") as f:
    whisper_texts = json.load(f)

In [ ]:
geo = [
    "france", "européen", "europe",
    "union européenne", "ue",
    "allemagne", "italie", "espagne",
    "états-unis", "chine", "russie"
]
institutions = [
    "république", "état", "gouvernement",
    "assemblée nationale", "sénat",
    "commission européenne", "conseil européen",
    "parlement européen"
]
parties = [
    "front national", "rassemblement national",
    "les républicains", "lr",
    "parti socialiste", "ps",
    "renaissance", "lrem"
]
persons = [
    "sarkozy", "bardella", "macron",
    "le pen", "marine le pen", "jean-marie le pen",
    "hollande", "chirac", "mitterrand"
]

In [ ]:
proper_names = persons + parties + institutions + geo

In [ ]:
books_sarkozy_clean = {
    name: preprocess_corpus(texts, proper_names=proper_names)
    for name, texts in books_sarkozy.items()
}

books_bardella_clean = {
    name: preprocess_corpus(texts, proper_names=proper_names)
    for name, texts in books_bardella.items()
}

whisper_bardella_clean = preprocess_corpus(
    whisper_texts,
    proper_names=proper_names
)

In [ ]:
whisper_bardella_clean

## découper les textes en blocks homogènes

In [ ]:
def text_to_blocks(text: str, block_size: int = 1000) -> list[str]:
    words = text.split()
    blocks = []
    for i in range(0, len(words), block_size):
        block = " ".join(words[i:i + block_size])
        if len(block.split()) >= block_size * 0.8:
            blocks.append(block)
    return blocks

In [ ]:
books_bardella_blocks = {}

for book_name, paragraphs in books_bardella_clean.items():
    full_text = " ".join(paragraphs)
    books_bardella_blocks[book_name] = text_to_blocks(full_text)

In [ ]:
full_text = ''

In [ ]:
books_sarkozy_blocks = {}

for book_name, paragraphs in books_sarkozy_clean.items():
    full_text = " ".join(paragraphs)
    books_sarkozy_blocks[book_name] = text_to_blocks(full_text)

In [ ]:
oral_bardella_blocks = text_to_blocks(
    " ".join(whisper_bardella_clean)
)

In [ ]:
books_bardella_blocks.keys()

In [ ]:
books_sarkozy_blocks.keys()

In [ ]:
oral_bardella_blocks

### on sauvegarde en vitesse

In [ ]:
BASE_DIR = Path(
    "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/blocks2"
)

BASE_DIR.mkdir(parents=True, exist_ok=True)
BASE_DIR

def save_json(data, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
sarkozy_blocks_json = {
    "source": "livres_sarkozy",
    "books": books_sarkozy_blocks
}

save_json(
    sarkozy_blocks_json,
    BASE_DIR / "blocks_livres_sarkozy.json"
)

bardella_blocks_json = {
    "source": "livres_bardella",
    "books": books_bardella_blocks
}

save_json(
    bardella_blocks_json,
    BASE_DIR / "blocks_livres_bardella.json"
)

oral_blocks_json = {
    "source": "oral_whisper_bardella",
    "blocks": oral_bardella_blocks
}

save_json(
    oral_blocks_json,
    BASE_DIR / "blocks_oral_bardella.json"
)

# ON COMMENCE LA STYLOM2TRIE OUI

### imports blocks

In [ ]:
folder_blocks = "/Users/mathieu/Documents/CODE/histosef_codes/bardella/data/blocks"


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
char_vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(3, 4),
    min_df=2,          # évite le bruit ultra-rare
    max_df=0.9,        # évite les artefacts trop communs
    lowercase=False    # déjà normalisé en amont
)

In [ ]:
FUNCTION_WORDS = [
    "et", "mais", "ou", "donc", "or", "car",
    "ainsi", "cependant", "pourtant",
    "alors", "enfin",
    "je", "nous", "on", "ils", "elles",
    "ce", "cela", "ça", "qui", "que", "dont",
    "ne", "pas", "plus", "jamais"
]

In [ ]:
def function_word_frequencies(text: str, vocab: list[str]) -> dict:
    words = text.split()
    total = len(words)
    freqs = {}

    for w in vocab:
        freqs[w] = words.count(w) / total if total > 0 else 0.0

    return freqs

In [ ]:
import pandas as pd

fw_rows = []

for block in X_texts:
    fw_rows.append(
        function_word_frequencies(block, FUNCTION_WORDS)
    )

X_fw = pd.DataFrame(fw_rows)

In [ ]:
from scipy.sparse import hstack

X_combined = hstack([
    X_char,
    X_fw.values
])

In [ ]:
labels = (
    ["livre"] * len(books_bardella_blocks["ce_que_veulent_bardella"])
    + ["oral"] * len(oral_bardella_blocks)
)